The algotithm ensures that each group first gets one high RL experience participant (mandatory) and then uses a greedy algorithm based on coding experience (secondary) to distribute the remaining participants as evenly as possible.

In [16]:
import pandas as pd
from collections import defaultdict
import numpy as np
import gender_guesser.detector as gender
from IPython.display import Markdown, display


def load_participants(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df['Do you want to take part in the RL challenge?'] == 'Yes']
    d = gender.Detector()
    df['sex'] = df['Name'].apply(lambda name: d.get_gender(str(name).split()[0]))
    return df

def compute_diversity_score(group):
    """
    Compute a simple diversity score based on counts of affiliation and inferred sex.
    Higher score means more diversity.
    """
    if len(group) <= 1:
        return 0
    affiliations = group['Affiliation'].value_counts(normalize=True)
    sexes = group['sex'].value_counts(normalize=True)
    affiliation_entropy = -sum(p * np.log(p) for p in affiliations)
    sex_entropy = -sum(p * np.log(p) for p in sexes)
    return affiliation_entropy + sex_entropy

def assign_groups(df, n_groups):
    # Separate participants based on RL experience.
    high_rl = df[df['Rate your RL experience (On an increasing scale of 1-5)'] >= 3]
    low_rl = df[df['Rate your RL experience (On an increasing scale of 1-5)'] < 4]

    if len(high_rl) < n_groups:
        raise ValueError("Not enough high-experience participants for number of groups")

    # Shuffle to randomize assignments.
    high_rl = high_rl.sample(frac=1, random_state=42).reset_index(drop=True)
    low_rl = low_rl.sample(frac=1, random_state=42).reset_index(drop=True)

    groups = defaultdict(list)
    coding_sums = [0] * n_groups  # Track total coding experience per group

    # Start each group with one high RL participant.
    for i in range(n_groups):
        participant = high_rl.iloc[i].to_dict()
        groups[i].append(participant)
        coding_sums[i] += participant['Rate your coding experience (On an increasing scale of 1-5)']

    # Remaining participants: rest of high RL and all low RL.
    remaining_high_rl = high_rl.iloc[n_groups:].to_dict(orient='records')
    remaining_low_rl = low_rl.to_dict(orient='records')
    remaining_participants = remaining_high_rl + remaining_low_rl

    # Sort remaining participants by coding experience descending for balanced distribution.
    remaining_participants.sort(key=lambda x: x['Rate your coding experience (On an increasing scale of 1-5)'], reverse=True)

    # Greedy assignment to the group with the lowest total coding experience.
    for participant in remaining_participants:
        min_group = min(range(n_groups), key=lambda i: coding_sums[i])
        groups[min_group].append(participant)
        coding_sums[min_group] += participant['Rate your coding experience (On an increasing scale of 1-5)']
    return groups

def groups_to_dataframe(groups):
    """
    Convert the groups dictionary into a single DataFrame.
    Each column represents a group, listing the participant names.
    If groups have different numbers of participants, missing values will be NaN.
    """
    # Determine the maximum number of participants in any group
    max_len = max(len(members) for members in groups.values())
    data = {}
    for group_id, members in groups.items():
        # For this example, we extract only the participant names.
        names = [member['Name'] for member in members]
        # Pad shorter lists with None
        names += [None] * (max_len - len(names))
        data[f'Group {group_id + 1}'] = names
    return pd.DataFrame(data)

In [17]:
# Example usage:

# Load participants from a CSV file (replace 'participants.csv' with your actual file path)

csv_path = "registrations_all.csv"
df = load_participants(csv_path)
df.remo

In [18]:
df.shape

(43, 10)

In [26]:
# Define number of groups you want to create
n_groups = 12  # For instance, 3 groups

# Assign groups based on the criteria in your function
groups = assign_groups(df, n_groups)

# Convert groups to a single DataFrame with groups as columns
groups_df = groups_to_dataframe(groups)

print(groups_df)

                    Group 1                Group 2             Group 3  \
0             Georg Schäfer        Sabrina Pochaba      Henry Day-Hall   
1           Julian Gethmann  Mohammad Sule Maidawa      Johan Leygonie   
2  Seyedeh Nasrin Mohammadi            Till Korten  Konrad Altenmüller   
3   Muhammad Abdullah Malik             Hannes Voß  Francesco Tripaldi   
4                      None                   None                None   

            Group 4                Group 5        Group 6         Group 7  \
0       Parth Patil           Penny Madysa  Leander Grech  Auralee Edelen   
1     Marco Bocchio       Farzad Jafarinia    Andre Dehne  Ibon Bustinduy   
2  Ferdinand Ferber            Parth Patil     Eya Dammak  Jason St. John   
3     Lorenz Fischl  Nicholas Tedjosantoso  Olga Mironova  Matthew Schwab   
4              None                   None           None            None   

          Group 8                Group 9          Group 10          Group 11  \
0      Joel 

In [27]:
# Export groups to CSV with a different name
all_participants = []
for group_id, members in groups.items():
    for participant in members:
        participant['group'] = group_id + 1  # Assign group number (starting from 1)
        all_participants.append(participant)
result_df = pd.DataFrame(all_participants)
result_df.to_csv('grouped_registrations_final.csv', index=False)
print('Exported grouped_registrations_v2.csv')


Exported grouped_registrations_v2.csv


In [28]:
groups

defaultdict(list,
            {0: [{'ID': 66,
               'Name': 'Georg Schäfer',
               'Email Address': 'georg.schaefer@fh-salzburg.ac.at',
               'Affiliation': 'Salzburg University of Applied Sciences',
               'Rate your RL experience (On an increasing scale of 1-5)': 3.0,
               'Rate your coding experience (On an increasing scale of 1-5)': 3.0,
               'Do you want to take part in the RL challenge?': 'Yes',
               'Registration state': 'Completed',
               'Tags': 'Lab Tour Group 1; Badge Printed; In Person',
               'sex': 'male',
               'group': 1},
              {'ID': 79,
               'Name': 'Julian Gethmann',
               'Email Address': 'julian.gethmann@kit.edu',
               'Affiliation': 'KIT',
               'Rate your RL experience (On an increasing scale of 1-5)': 1.0,
               'Rate your coding experience (On an increasing scale of 1-5)': 5.0,
               'Do you want to take pa

In [29]:
print(groups_df)

                    Group 1                Group 2             Group 3  \
0             Georg Schäfer        Sabrina Pochaba      Henry Day-Hall   
1           Julian Gethmann  Mohammad Sule Maidawa      Johan Leygonie   
2  Seyedeh Nasrin Mohammadi            Till Korten  Konrad Altenmüller   
3   Muhammad Abdullah Malik             Hannes Voß  Francesco Tripaldi   
4                      None                   None                None   

            Group 4                Group 5        Group 6         Group 7  \
0       Parth Patil           Penny Madysa  Leander Grech  Auralee Edelen   
1     Marco Bocchio       Farzad Jafarinia    Andre Dehne  Ibon Bustinduy   
2  Ferdinand Ferber            Parth Patil     Eya Dammak  Jason St. John   
3     Lorenz Fischl  Nicholas Tedjosantoso  Olga Mironova  Matthew Schwab   
4              None                   None           None            None   

          Group 8                Group 9          Group 10          Group 11  \
0      Joel 

In [30]:
print(groups_df.to_markdown())

|    | Group 1                  | Group 2               | Group 3            | Group 4          | Group 5               | Group 6       | Group 7        | Group 8        | Group 9               | Group 10         | Group 11         | Group 12              |
|---:|:-------------------------|:----------------------|:-------------------|:-----------------|:----------------------|:--------------|:---------------|:---------------|:----------------------|:-----------------|:-----------------|:----------------------|
|  0 | Georg Schäfer            | Sabrina Pochaba       | Henry Day-Hall     | Parth Patil      | Penny Madysa          | Leander Grech | Auralee Edelen | Joel Wulff     | M Asif Hasan          | Olga Mironova    | Hayg GULER       | Adrián Menor de Oñate |
|  1 | Julian Gethmann          | Mohammad Sule Maidawa | Johan Leygonie     | Marco Bocchio    | Farzad Jafarinia      | Andre Dehne   | Ibon Bustinduy | Penny Madysa   | Randeer Pratap Gautam | Juan Luis Muñoz  | Amelia Poll

In [31]:
# Assuming groups_df is your DataFrame with groups as columns:
md_table = groups_df.to_markdown(index=False)
display(Markdown(md_table))

| Group 1                  | Group 2               | Group 3            | Group 4          | Group 5               | Group 6       | Group 7        | Group 8        | Group 9               | Group 10         | Group 11         | Group 12              |
|:-------------------------|:----------------------|:-------------------|:-----------------|:----------------------|:--------------|:---------------|:---------------|:----------------------|:-----------------|:-----------------|:----------------------|
| Georg Schäfer            | Sabrina Pochaba       | Henry Day-Hall     | Parth Patil      | Penny Madysa          | Leander Grech | Auralee Edelen | Joel Wulff     | M Asif Hasan          | Olga Mironova    | Hayg GULER       | Adrián Menor de Oñate |
| Julian Gethmann          | Mohammad Sule Maidawa | Johan Leygonie     | Marco Bocchio    | Farzad Jafarinia      | Andre Dehne   | Ibon Bustinduy | Penny Madysa   | Randeer Pratap Gautam | Juan Luis Muñoz  | Amelia Pollard   | Gerhard Hejc          |
| Seyedeh Nasrin Mohammadi | Till Korten           | Konrad Altenmüller | Ferdinand Ferber | Parth Patil           | Eya Dammak    | Jason St. John | Hayg GULER     | Georg Schäfer         | Pardis Niknejadi | Thorsten Hellert | Sebastian Starke      |
| Muhammad Abdullah Malik  | Hannes Voß            | Francesco Tripaldi | Lorenz Fischl    | Nicholas Tedjosantoso | Olga Mironova | Matthew Schwab | Stefano Krecic | Adrián Menor de Oñate | Gesa Goetzke     | Amna Majid       | Nadezhda Khachatrian  |
|                          |                       |                    |                  |                       |               |                | Bindu Sharan   |                       |                  |                  |                       |